In [1]:
# setup

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset as TorchDataset
from tqdm import tqdm
from pathlib import Path
import random
import json
import matplotlib.pyplot as plt

OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print(f"Device: {DEVICE}")

preferred_files = ["books_big_nli.parquet", "books_big.parquet"]
DATA_PATH = None
for fname in preferred_files:
    candidates = list(Path("/kaggle/input").rglob(fname))
    if candidates:
        DATA_PATH = candidates[0]
        break

if DATA_PATH is None:
    raise FileNotFoundError("Could not find books_big_nli.parquet or books_big.parquet in /kaggle/input")

print(f"Using data file: {DATA_PATH}")
df = pd.read_parquet(DATA_PATH)

print(f"Rows: {len(df):,} | Users: {df['user_id'].nunique():,} | Items: {df['item_id'].nunique():,}")
print("Columns:", df.columns.tolist())

RAW_COLS = ["nli_raw_plot", "nli_raw_characters", "nli_raw_writing"]
CTR_COLS = ["nli_ctr_plot", "nli_ctr_characters", "nli_ctr_writing"]
required_cols = RAW_COLS + CTR_COLS

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing NLI columns: {missing}")

# user/item mappings
user2idx = {uid: i for i, uid in enumerate(df["user_id"].unique())}
item2idx = {iid: i for i, iid in enumerate(df["item_id"].unique())}
idx2item = {v: k for k, v in item2idx.items()}

n_users = len(user2idx)
n_items = len(item2idx)

df["user_idx"] = df["user_id"].map(user2idx)
df["item_idx"] = df["item_id"].map(item2idx)

# leave-one-out split
df = df.sort_values(["user_id", "timestamp"]).copy()
df["_rank"] = df.groupby("user_id")["timestamp"].rank(method="first", ascending=False)

test_df = df[df["_rank"] == 1].copy()
val_df  = df[df["_rank"] == 2].copy()
train_df = df[df["_rank"] > 2].copy()
df = df.drop(columns=["_rank"])

val_gt  = val_df.groupby("user_id")["item_id"].apply(set).to_dict()
test_gt = test_df.groupby("user_id")["item_id"].apply(set).to_dict()

seen_items_val  = train_df.groupby("user_id")["item_idx"].apply(set).to_dict()
seen_items_test = pd.concat([train_df, val_df]).groupby("user_id")["item_idx"].apply(set).to_dict()

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

Device: cuda
Using data file: /kaggle/input/datasets/rita12390/nli-files/books_big_nli.parquet
Rows: 563,929 | Users: 32,709 | Items: 2,000
Columns: ['user_id', 'item_id', 'rating', 'timestamp', 'year', 'verified', 'item_format', 'text_combined', 'review_len_words', 'sentiment_score', 'aspect_plot', 'aspect_characters', 'aspect_writing', 'aspect_pacing', 'aspect_ending', 'aspect_overall', 'nli_raw_plot', 'nli_raw_characters', 'nli_raw_writing', 'nli_ctr_plot', 'nli_ctr_characters', 'nli_ctr_writing']
Train: 498,511 | Val: 32,709 | Test: 32,709


In [2]:
# item-level NLI feature tensors

RAW_COLS = ["nli_raw_plot", "nli_raw_characters", "nli_raw_writing"]
CTR_COLS = ["nli_ctr_plot", "nli_ctr_characters", "nli_ctr_writing"]

missing = [c for c in RAW_COLS + CTR_COLS if c not in df.columns]
if missing:
    raise ValueError(f"Missing NLI columns: {missing}")

# one item-level row per item
item_df = (
    df.sort_values(["item_idx", "timestamp"])
      .groupby("item_idx")[RAW_COLS + CTR_COLS]
      .first()
      .reindex(range(n_items))
)

def build_feature_tensor(item_df, cols):
    values = item_df[cols].to_numpy(dtype=np.float32)
    mask = ~np.isnan(values)

    # z-score on observed values only
    for j in range(values.shape[1]):
        m = mask[:, j]
        if m.sum() > 0:
            mu = values[m, j].mean()
            sd = values[m, j].std() + 1e-8
            values[m, j] = (values[m, j] - mu) / sd
        values[~m, j] = 0.0

    feat_tensor = torch.tensor(values, dtype=torch.float32, device=DEVICE)
    mask_tensor = torch.tensor(mask, dtype=torch.bool, device=DEVICE)
    return feat_tensor, mask_tensor

item_feat_raw, item_mask_raw = build_feature_tensor(item_df, RAW_COLS)
item_feat_ctr, item_mask_ctr = build_feature_tensor(item_df, CTR_COLS)

print("Raw feature tensor:", item_feat_raw.shape)
print("Centered feature tensor:", item_feat_ctr.shape)

print("\nCoverage by raw feature:")
for col in RAW_COLS:
    cov = item_df[col].notna().mean() * 100
    print(f"  {col}: {cov:.1f}%")

print("\nCoverage by centered feature:")
for col in CTR_COLS:
    cov = item_df[col].notna().mean() * 100
    print(f"  {col}: {cov:.1f}%")

Raw feature tensor: torch.Size([2000, 3])
Centered feature tensor: torch.Size([2000, 3])

Coverage by raw feature:
  nli_raw_plot: 99.7%
  nli_raw_characters: 99.4%
  nli_raw_writing: 100.0%

Coverage by centered feature:
  nli_ctr_plot: 99.7%
  nli_ctr_characters: 99.4%
  nli_ctr_writing: 100.0%


In [3]:
# metrics, dataset, model

def ndcg_at_k(recommended, relevant, k):
    if not relevant:
        return 0.0
    dcg = sum(1.0 / np.log2(i + 2) for i, item in enumerate(recommended[:k]) if item in relevant)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(relevant), k)))
    return dcg / idcg if idcg > 0 else 0.0

def recall_at_k(recommended, relevant, k):
    if not relevant:
        return 0.0
    return len(set(recommended[:k]) & relevant) / len(relevant)

def precision_at_k(recommended, relevant, k):
    return len(set(recommended[:k]) & relevant) / k

def hr_at_k(recommended, relevant, k):
    return 1.0 if set(recommended[:k]) & relevant else 0.0

def evaluate_recommendations(recs, ground_truth, k=10):
    ndcgs, recalls, precisions, hrs = [], [], [], []
    for uid, rec_list in recs.items():
        gt = ground_truth.get(uid, set())
        if not gt:
            continue
        ndcgs.append(ndcg_at_k(rec_list, gt, k))
        recalls.append(recall_at_k(rec_list, gt, k))
        precisions.append(precision_at_k(rec_list, gt, k))
        hrs.append(hr_at_k(rec_list, gt, k))
    return {
        f"NDCG@{k}": float(np.mean(ndcgs)),
        f"Recall@{k}": float(np.mean(recalls)),
        f"Precision@{k}": float(np.mean(precisions)),
        f"HR@{k}": float(np.mean(hrs)),
        "n_users_evaluated": len(ndcgs),
    }

class BPRDataset(TorchDataset):
    def __init__(self, df, n_items):
        df = df[(df["user_idx"] >= 0) & (df["item_idx"] >= 0)].copy()
        self.users = df["user_idx"].values
        self.pos_items = df["item_idx"].values
        self.n_items = n_items
        self.user_items = df.groupby("user_idx")["item_idx"].apply(set).to_dict()

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        user = self.users[idx]
        pos = self.pos_items[idx]
        seen = self.user_items.get(user, set())

        neg = random.randint(0, self.n_items - 1)
        while neg in seen:
            neg = random.randint(0, self.n_items - 1)

        return (
            torch.tensor(user, dtype=torch.long),
            torch.tensor(pos, dtype=torch.long),
            torch.tensor(neg, dtype=torch.long),
        )

# aspect-CF with user-conditioned attention over RAW NLI aspects only
class AspectCF_Raw(nn.Module):
    def __init__(
        self,
        n_users,
        n_items,
        n_raw=3,
        emb_dim_gmf=32,
        emb_dim_mlp=32,
        aspect_dim=16,
        mlp_layers=None,
        dropout=0.1,
    ):
        super().__init__()
        if mlp_layers is None:
            mlp_layers = [64, 32, 16]

        self.n_raw = n_raw

        self.gmf_user = nn.Embedding(n_users, emb_dim_gmf)
        self.gmf_item = nn.Embedding(n_items, emb_dim_gmf)

        self.mlp_user = nn.Embedding(n_users, emb_dim_mlp)
        self.mlp_item = nn.Embedding(n_items, emb_dim_mlp)

        self.user_query = nn.Embedding(n_users, aspect_dim)
        self.raw_token_emb = nn.Embedding(n_raw, aspect_dim)
        self.raw_proj = nn.Linear(aspect_dim, emb_dim_mlp)
        self.fusion_norm = nn.LayerNorm(emb_dim_mlp)

        mlp_input = emb_dim_mlp * 2
        layers = []
        for out_dim in mlp_layers:
            layers += [nn.Linear(mlp_input, out_dim), nn.ReLU(), nn.Dropout(dropout)]
            mlp_input = out_dim
        self.mlp = nn.Sequential(*layers)

        self.output_layer = nn.Linear(emb_dim_gmf + mlp_layers[-1], 1)
        self._init_weights()

    def _init_weights(self):
        for emb in [self.gmf_user, self.gmf_item, self.mlp_user, self.mlp_item, self.user_query, self.raw_token_emb]:
            nn.init.normal_(emb.weight, std=0.01)

        nn.init.xavier_uniform_(self.raw_proj.weight)
        if self.raw_proj.bias is not None:
            nn.init.zeros_(self.raw_proj.bias)

        for m in self.mlp:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

        nn.init.xavier_uniform_(self.output_layer.weight)
        if self.output_layer.bias is not None:
            nn.init.zeros_(self.output_layer.bias)

    def forward(self, user_idx, item_idx, item_feat_raw, item_mask_raw):
        gmf_out = self.gmf_user(user_idx) * self.gmf_item(item_idx)

        raw_vals = item_feat_raw[item_idx]  # (B, 3)
        raw_mask = item_mask_raw[item_idx]  # (B, 3)

        token_ids = torch.arange(self.n_raw, device=raw_vals.device)
        token_base = self.raw_token_emb(token_ids)                                # (3, A)
        token_base = token_base.unsqueeze(0).expand(raw_vals.shape[0], -1, -1)    # (B, 3, A)

        token_repr = raw_vals.unsqueeze(-1) * token_base                          # (B, 3, A)
        q = self.user_query(user_idx).unsqueeze(1)                                # (B, 1, A)

        scores = (token_repr * q).sum(dim=-1) / np.sqrt(token_repr.shape[-1])
        scores = scores.masked_fill(~raw_mask, -1e9)

        attn = torch.softmax(scores, dim=-1)
        valid_any = raw_mask.any(dim=1, keepdim=True)
        attn = torch.where(valid_any, attn, torch.zeros_like(attn))

        raw_out = (attn.unsqueeze(-1) * token_repr).sum(dim=1)                    # (B, A)
        raw_out = self.raw_proj(raw_out)                                          # (B, emb_dim_mlp)
        raw_out = self.fusion_norm(raw_out)

        item_fused = self.mlp_item(item_idx) + raw_out
        mlp_in = torch.cat([self.mlp_user(user_idx), item_fused], dim=-1)
        mlp_out = self.mlp(mlp_in)

        combined = torch.cat([gmf_out, mlp_out], dim=-1)
        return self.output_layer(combined).squeeze(-1)

class AspectCF_RawCtr(nn.Module):
    """
    Aspect-CF v2:
      - RAW branch: user-conditioned attention over raw aspects
      - CTR branch: separate signed linear projection of centered features
      - gated fusion of raw and centered branches
    """
    def __init__(
        self,
        n_users,
        n_items,
        n_raw=3,
        n_ctr=3,
        emb_dim_gmf=32,
        emb_dim_mlp=32,
        aspect_dim=16,
        mlp_layers=None,
        dropout=0.1,
    ):
        super().__init__()
        if mlp_layers is None:
            mlp_layers = [64, 32, 16]

        self.n_raw = n_raw
        self.n_ctr = n_ctr

        self.gmf_user = nn.Embedding(n_users, emb_dim_gmf)
        self.gmf_item = nn.Embedding(n_items, emb_dim_gmf)

        self.mlp_user = nn.Embedding(n_users, emb_dim_mlp)
        self.mlp_item = nn.Embedding(n_items, emb_dim_mlp)

        # raw branch
        self.user_query = nn.Embedding(n_users, aspect_dim)
        self.raw_token_emb = nn.Embedding(n_raw, aspect_dim)
        self.raw_proj = nn.Linear(aspect_dim, emb_dim_mlp)

        # centered branch: signed linear transform, no ReLU
        self.ctr_proj = nn.Linear(n_ctr, emb_dim_mlp)

        # scalar fusion gate
        self.gate_logit = nn.Parameter(torch.tensor(0.0))
        self.fusion_norm = nn.LayerNorm(emb_dim_mlp)

        mlp_input = emb_dim_mlp * 2
        layers = []
        for out_dim in mlp_layers:
            layers += [nn.Linear(mlp_input, out_dim), nn.ReLU(), nn.Dropout(dropout)]
            mlp_input = out_dim
        self.mlp = nn.Sequential(*layers)

        self.output_layer = nn.Linear(emb_dim_gmf + mlp_layers[-1], 1)
        self._init_weights()

    def _init_weights(self):
        for emb in [self.gmf_user, self.gmf_item, self.mlp_user, self.mlp_item, self.user_query, self.raw_token_emb]:
            nn.init.normal_(emb.weight, std=0.01)

        for layer in [self.raw_proj, self.ctr_proj]:
            nn.init.xavier_uniform_(layer.weight)
            if layer.bias is not None:
                nn.init.zeros_(layer.bias)

        for m in self.mlp:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

        nn.init.xavier_uniform_(self.output_layer.weight)
        if self.output_layer.bias is not None:
            nn.init.zeros_(self.output_layer.bias)

    def forward(self, user_idx, item_idx, item_feat_raw, item_mask_raw, item_feat_ctr):
        gmf_out = self.gmf_user(user_idx) * self.gmf_item(item_idx)

        # raw branch
        raw_vals = item_feat_raw[item_idx]  # (B, 3)
        raw_mask = item_mask_raw[item_idx]  # (B, 3)

        token_ids = torch.arange(self.n_raw, device=raw_vals.device)
        token_base = self.raw_token_emb(token_ids)                                # (3, A)
        token_base = token_base.unsqueeze(0).expand(raw_vals.shape[0], -1, -1)    # (B, 3, A)

        token_repr = raw_vals.unsqueeze(-1) * token_base                          # (B, 3, A)
        q = self.user_query(user_idx).unsqueeze(1)                                # (B, 1, A)

        scores = (token_repr * q).sum(dim=-1) / np.sqrt(token_repr.shape[-1])
        scores = scores.masked_fill(~raw_mask, -1e9)

        attn = torch.softmax(scores, dim=-1)
        valid_any = raw_mask.any(dim=1, keepdim=True)
        attn = torch.where(valid_any, attn, torch.zeros_like(attn))

        raw_out = (attn.unsqueeze(-1) * token_repr).sum(dim=1)                    # (B, A)
        raw_out = self.raw_proj(raw_out)                                          # (B, emb_dim_mlp)

        # centered branch
        ctr_vals = item_feat_ctr[item_idx]                                        # (B, 3)
        ctr_out = self.ctr_proj(ctr_vals)                                         # (B, emb_dim_mlp)

        # gated fusion
        gate = torch.sigmoid(self.gate_logit)                                     # scalar in (0,1)
        fused_aspect = gate * raw_out + (1.0 - gate) * ctr_out
        fused_aspect = self.fusion_norm(fused_aspect)

        item_fused = self.mlp_item(item_idx) + fused_aspect
        mlp_in = torch.cat([self.mlp_user(user_idx), item_fused], dim=-1)
        mlp_out = self.mlp(mlp_in)

        combined = torch.cat([gmf_out, mlp_out], dim=-1)
        return self.output_layer(combined).squeeze(-1)


In [4]:
# config

EMB_DIM_GMF = 32
EMB_DIM_MLP = 32
ASPECT_DIM = 16
MLP_LAYERS = [64, 32, 16]
DROPOUT = 0.1
LR = 1e-3
WEIGHT_DECAY = 1e-5
N_EPOCHS = 50
BATCH_SIZE = 2048
PATIENCE = 4
EVAL_EVERY = 2
K = 10

dataset = BPRDataset(train_df, n_items)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
all_item_idxs = np.arange(n_items)

VAL_USERS_FIXED = list(val_gt.keys())
if len(VAL_USERS_FIXED) > 3000:
    random.seed(SEED)
    VAL_USERS_FIXED = random.sample(VAL_USERS_FIXED, 3000)

print(f"Fixed quick-val subset: {len(VAL_USERS_FIXED):,} users")

def train_and_evaluate(model, model_name, score_fn):
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    best_ndcg = -1.0
    no_improve = 0
    best_state = None
    history = []

    print(f"\nTraining {model_name}...")
    print(f"Params: {sum(p.numel() for p in model.parameters()):,}\n")

    for epoch in range(1, N_EPOCHS + 1):
        model.train()
        total_loss = 0.0

        for users_b, pos_b, neg_b in tqdm(loader, desc=f"Epoch {epoch}", leave=False):
            users_b = users_b.to(DEVICE)
            pos_b = pos_b.to(DEVICE)
            neg_b = neg_b.to(DEVICE)

            pos_score = score_fn(model, users_b, pos_b)
            neg_score = score_fn(model, users_b, neg_b)
            loss = -torch.log(torch.sigmoid(pos_score - neg_score) + 1e-8).mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(users_b)

        avg_loss = total_loss / len(dataset)

        if epoch % EVAL_EVERY == 0 or epoch == 1:
            model.eval()
            users = VAL_USERS_FIXED
            scores_list = []

            with torch.no_grad():
                all_items_t = torch.tensor(all_item_idxs, dtype=torch.long, device=DEVICE)
                for uid in users:
                    uidx = user2idx.get(uid, -1)
                    if uidx < 0:
                        continue

                    user_t = torch.full((n_items,), uidx, dtype=torch.long, device=DEVICE)
                    sc = score_fn(model, user_t, all_items_t).cpu().numpy()

                    for sidx in seen_items_val.get(uid, set()):
                        sc[sidx] = -1e9

                    top = np.argsort(sc)[::-1][:K]
                    rec = [idx2item[i] for i in top]
                    scores_list.append(ndcg_at_k(rec, val_gt[uid], K))

            val_ndcg = float(np.mean(scores_list)) if scores_list else 0.0
            print(f"  Epoch {epoch:>3}/{N_EPOCHS}  BPR: {avg_loss:.4f}  val_NDCG@{K}: {val_ndcg:.4f}")
            history.append({"epoch": epoch, "loss": avg_loss, "val_ndcg": val_ndcg})

            if val_ndcg > best_ndcg + 1e-5:
                best_ndcg = val_ndcg
                no_improve = 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                no_improve += 1
                if no_improve >= PATIENCE:
                    print(f"\n  Early stopping at epoch {epoch} (best: {best_ndcg:.4f})")
                    break

    if best_state is not None:
        model.load_state_dict(best_state)
        model.to(DEVICE)

    print(f"  Best val NDCG@{K}: {best_ndcg:.4f}")

    @torch.no_grad()
    def generate_recs(target_users, seen_dict):
        model.eval()
        all_items_t = torch.tensor(all_item_idxs, dtype=torch.long, device=DEVICE)
        recs = {}

        for uid in tqdm(target_users, desc="Recs", leave=False):
            uidx = user2idx.get(uid, -1)
            if uidx < 0:
                continue

            user_t = torch.full((n_items,), uidx, dtype=torch.long, device=DEVICE)
            sc = score_fn(model, user_t, all_items_t).cpu().numpy()

            for sidx in seen_dict.get(uid, set()):
                sc[sidx] = -1e9

            top = np.argsort(sc)[::-1][:K]
            recs[uid] = [idx2item[i] for i in top]

        return recs

    print(f"\nEvaluating {model_name} on VAL...")
    val_recs = generate_recs(val_df["user_id"].unique().tolist(), seen_items_val)
    val_metrics = evaluate_recommendations(val_recs, val_gt, k=K)

    print(f"Evaluating {model_name} on TEST...")
    test_recs = generate_recs(test_df["user_id"].unique().tolist(), seen_items_test)
    test_metrics = evaluate_recommendations(test_recs, test_gt, k=K)

    print(f"\n{'='*40}")
    print(f"{model_name} — VAL")
    print(f"{'='*40}")
    for key, val in val_metrics.items():
        print(f"  {key:<20} {val:.4f}" if isinstance(val, float) else f"  {key:<20} {val}")

    print(f"\n{model_name} — TEST")
    print(f"{'='*40}")
    for key, val in test_metrics.items():
        print(f"  {key:<20} {val:.4f}" if isinstance(val, float) else f"  {key:<20} {val}")

    fname = model_name.lower().replace("+", "_").replace("-", "_").replace(" ", "_")
    results = {"model": model_name, "val": val_metrics, "test": test_metrics}

    with open(OUT_DIR / f"{fname}_results.json", "w") as f:
        json.dump(results, f, indent=2)
    torch.save(model.state_dict(), OUT_DIR / f"{fname}_model.pt")
    pd.DataFrame(history).to_csv(OUT_DIR / f"{fname}_history.csv", index=False)

    print(f"Saved: {fname}_results.json, {fname}_model.pt, {fname}_history.csv")
    return val_metrics, test_metrics, history


Fixed quick-val subset: 3,000 users


In [5]:
# train aspect-cf (nli-raw)

model_aspect_raw = AspectCF_Raw(
    n_users=n_users,
    n_items=n_items,
    n_raw=3,
    emb_dim_gmf=EMB_DIM_GMF,
    emb_dim_mlp=EMB_DIM_MLP,
    aspect_dim=ASPECT_DIM,
    mlp_layers=MLP_LAYERS,
    dropout=DROPOUT,
).to(DEVICE)

def score_aspect_raw(model, user_idx, item_idx):
    return model(user_idx, item_idx, item_feat_raw, item_mask_raw)

aspect_raw_val, aspect_raw_test, aspect_raw_hist = train_and_evaluate(
    model_aspect_raw,
    "Aspect-CF (NLI raw)",
    score_aspect_raw,
)



Training Aspect-CF (NLI raw)...
Params: 2,752,193



  Epoch   1/50  BPR: 0.6771  val_NDCG@10: 0.0038


  Epoch   2/50  BPR: 0.5238  val_NDCG@10: 0.0060


  Epoch   4/50  BPR: 0.3515  val_NDCG@10: 0.0111


  Epoch   6/50  BPR: 0.2929  val_NDCG@10: 0.0150


...

  Epoch  46/50  BPR: 0.0971  val_NDCG@10: 0.0451


  Epoch  48/50  BPR: 0.0960  val_NDCG@10: 0.0485


  Epoch  50/50  BPR: 0.0933  val_NDCG@10: 0.0475
  Best val NDCG@10: 0.0485

Evaluating Aspect-CF (NLI raw) on VAL...


Evaluating Aspect-CF (NLI raw) on TEST...



Aspect-CF (NLI raw) — VAL
  NDCG@10              0.0476
  Recall@10            0.0941
  Precision@10         0.0094
  HR@10                0.0941
  n_users_evaluated    32709

Aspect-CF (NLI raw) — TEST
  NDCG@10              0.0301
  Recall@10            0.0618
  Precision@10         0.0062
  HR@10                0.0618
  n_users_evaluated    32709
Saved: aspect_cf_(nli_raw)_results.json, aspect_cf_(nli_raw)_model.pt, aspect_cf_(nli_raw)_history.csv


In [6]:
# train aspect-cf (nli raw + ctr v2)

model_aspect_rawctr = AspectCF_RawCtr(
    n_users=n_users,
    n_items=n_items,
    n_raw=3,
    n_ctr=3,
    emb_dim_gmf=EMB_DIM_GMF,
    emb_dim_mlp=EMB_DIM_MLP,
    aspect_dim=ASPECT_DIM,
    mlp_layers=MLP_LAYERS,
    dropout=DROPOUT,
).to(DEVICE)

def score_aspect_rawctr(model, user_idx, item_idx):
    return model(user_idx, item_idx, item_feat_raw, item_mask_raw, item_feat_ctr)

aspect_rawctr_val, aspect_rawctr_test, aspect_rawctr_hist = train_and_evaluate(
    model_aspect_rawctr,
    "Aspect-CF (NLI raw+ctr v2)",
    score_aspect_rawctr,
)



Training Aspect-CF (NLI raw+ctr v2)...
Params: 2,752,322



  Epoch   1/50  BPR: 0.6587  val_NDCG@10: 0.0037


  Epoch   2/50  BPR: 0.4764  val_NDCG@10: 0.0086


  Epoch   4/50  BPR: 0.3296  val_NDCG@10: 0.0152


  Epoch   6/50  BPR: 0.2737  val_NDCG@10: 0.0159


...

  Epoch  42/50  BPR: 0.1014  val_NDCG@10: 0.0431


  Epoch  44/50  BPR: 0.0984  val_NDCG@10: 0.0442


  Epoch  46/50  BPR: 0.0957  val_NDCG@10: 0.0393

  Early stopping at epoch 46 (best: 0.0449)
  Best val NDCG@10: 0.0449

Evaluating Aspect-CF (NLI raw+ctr v2) on VAL...


Evaluating Aspect-CF (NLI raw+ctr v2) on TEST...



Aspect-CF (NLI raw+ctr v2) — VAL
  NDCG@10              0.0438
  Recall@10            0.0846
  Precision@10         0.0085
  HR@10                0.0846
  n_users_evaluated    32709

Aspect-CF (NLI raw+ctr v2) — TEST
  NDCG@10              0.0283
  Recall@10            0.0562
  Precision@10         0.0056
  HR@10                0.0562
  n_users_evaluated    32709
Saved: aspect_cf_(nli_raw_ctr_v2)_results.json, aspect_cf_(nli_raw_ctr_v2)_model.pt, aspect_cf_(nli_raw_ctr_v2)_history.csv
